In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# Load the cleaned dataset
df = pd.read_csv("df_cleaned.csv")
TARGET_COL = "phase"

print("Data Loaded. Shape:", df.shape)

Data Loaded. Shape: (2970, 61)


In [3]:
df['phase'].value_counts()

phase
Luteal        953
Follicular    805
Fertility     656
Menstrual     556
Name: count, dtype: int64

### Baseline Model + XGBoost Tuning

In [2]:
print("=== TUNING MODEL 1: XGBOOST (BASELINE) ===")

# 1. Define Baseline Data (Remove Collinear Cols)
drop_cols = [
    "composition_score", "revitalization_score", "duration_score", 
    "glucose_mean", "glucose_min", "glucose_std", 
    "minutesasleep", "duration_exercise", "calories_exercise"
]
# Ensure we only drop columns that actually exist
cols_to_drop = [c for c in drop_cols if c in df.columns]
df_baseline = df.drop(columns=cols_to_drop)

X = df_baseline.drop(columns=[TARGET_COL])
y = df_baseline[TARGET_COL]

# 2. Preprocessing Pipeline
# Identify numeric and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

# Define Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# 3. Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 4. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 5. Define XGBoost Pipeline
# --- FIX: Removed 'use_label_encoder=False' to stop warnings ---
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=42
    ))
])

# 6. Define Hyperparameter Grid
param_grid_xgb = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [3, 4, 5],
    'classifier__subsample': [0.8, 1.0]
}

# 7. Run GridSearchCV
grid_xgb = GridSearchCV(
    xgb_pipeline, 
    param_grid_xgb, 
    cv=5, 
    scoring='accuracy', 
    verbose=1, 
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

# 8. Results
print(f"\nBest XGBoost Params: {grid_xgb.best_params_}")
print(f"Best CV Accuracy: {grid_xgb.best_score_:.4f}")

# Final Test Evaluation
y_pred = grid_xgb.predict(X_test)
print("\nTest Set Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

=== TUNING MODEL 1: XGBOOST (BASELINE) ===
Fitting 5 folds for each of 54 candidates, totalling 270 fits

Best XGBoost Params: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 5, 'classifier__n_estimators': 300, 'classifier__subsample': 0.8}
Best CV Accuracy: 0.6625

Test Set Accuracy: 0.6986531986531986
              precision    recall  f1-score   support

   Fertility       0.65      0.53      0.59       131
  Follicular       0.69      0.73      0.71       161
      Luteal       0.69      0.80      0.74       191
   Menstrual       0.77      0.68      0.72       111

    accuracy                           0.70       594
   macro avg       0.70      0.68      0.69       594
weighted avg       0.70      0.70      0.70       594



### Random Forest + Forward Selection Tuning

In [6]:
print("\n=== TUNING MODEL 2: RANDOM FOREST (FORWARD SELECTION) ===")

# 1. Define Forward Selection Features (Excluding day_in_study as requested)
# Note: 'phase' is the target, so we don't include it in the feature list
forward_features = [
    'lh', 'estrogen', 'calories_daily', 'temp_diff_from_baseline_max', 
    'glucose_mean', 'glucose_std', 'revitalization_score', 
    'minutestofallasleep', 'minutesasleep', 'minutesafterwakeup', 
    'duration_exercise', 'calories_exercise', 'heartrate_exercise', 
    'in_default_zone_3', 'in_default_zone_2', 'flow_volume_score', 
    'appetite_score', 'cramps_score', 'sorebreasts_score', 'fatigue_score', 
    'sleepissue_score', 'foodcravings_score', 'flow_color', 'activity_type', 
    'gender', 'ethnicity', 'education', 'sexually_active', 
    'self_report_menstrual_health_literacy'
]

# Select only these columns + Target
X = df[forward_features].copy()
y = df[TARGET_COL]

# 2. Preprocessing (Same logic, new subset)
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# 3. Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 4. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 5. Define Random Forest Pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# 6. Define Hyperparameter Grid
param_grid_rf = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

# 7. Run GridSearchCV
grid_rf = GridSearchCV(
    rf_pipeline, 
    param_grid_rf, 
    cv=5, 
    scoring='accuracy', 
    verbose=1, 
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

# 8. Results
print(f"\nBest Random Forest Params: {grid_rf.best_params_}")
print(f"Best CV Accuracy: {grid_rf.best_score_:.4f}")

# Final Test Evaluation
y_pred = grid_rf.predict(X_test)
print("\nTest Set Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))


=== TUNING MODEL 2: RANDOM FOREST (FORWARD SELECTION) ===
Fitting 5 folds for each of 81 candidates, totalling 405 fits

Best Random Forest Params: {'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 300}
Best CV Accuracy: 0.6225

Test Set Accuracy: 0.6565656565656566
              precision    recall  f1-score   support

   Fertility       0.68      0.44      0.54       131
  Follicular       0.66      0.58      0.62       161
      Luteal       0.62      0.88      0.73       191
   Menstrual       0.74      0.64      0.69       111

    accuracy                           0.66       594
   macro avg       0.67      0.63      0.64       594
weighted avg       0.67      0.66      0.65       594



In [9]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

# 1. Load Data
df = pd.read_csv("df_cleaned.csv")
TARGET_COL = "phase"

# 2. Define Feature Subsets
# --- Baseline (All Features minus High VIF) ---
drop_cols_baseline = [
    "composition_score", "revitalization_score", "duration_score", 
    "glucose_mean", "glucose_min", "glucose_std", 
    "minutesasleep", "duration_exercise", "calories_exercise"
]
baseline_cols = [c for c in df.columns if c not in drop_cols_baseline and c != TARGET_COL]

# --- Forward Selection ---
forward_features = [
    'lh', 'estrogen', 'calories_daily', 'temp_diff_from_baseline_max', 
    'glucose_mean', 'glucose_std', 'revitalization_score', 
    'minutestofallasleep', 'minutesasleep', 'minutesafterwakeup', 
    'duration_exercise', 'calories_exercise', 'heartrate_exercise', 
    'in_default_zone_3', 'in_default_zone_2', 'flow_volume_score', 
    'appetite_score', 'cramps_score', 'sorebreasts_score', 'fatigue_score', 
    'sleepissue_score', 'foodcravings_score', 'flow_color', 'activity_type', 
    'gender', 'ethnicity', 'education', 'sexually_active', 
    'self_report_menstrual_health_literacy'
]

# --- Backward Selection ---
backward_features = [
    'lh','estrogen','bpm_mean','bpm_min',
    'temp_diff_from_baseline_max', 'nightly_temperature',
    'efficiency', 'in_default_zone_2', 'below_default_zone_1',
    'age_of_first_menarche', 'flow_volume_score',
    'exerciselevel_score', 'cramps_score', 'sorebreasts_score',
    'flow_color'
]

# --- ANOVA ---
anova_features = [
    'flow_volume_score', 'cramps_score', 'lh', 'estrogen', 'flow_color', 
    'sorebreasts_score', 'nightly_temperature', 'bpm_min', 'bpm_mean', 
    'bloating_score', 'temp_diff_from_baseline_max', 'revitalization_score', 
    'in_default_zone_1', 'indigestion_score', 'headaches_score'
]

# --- SelectKBest ---
kbest_features = [
    'lh', 'estrogen' ,'bpm_min', 'nightly_temperature',
    'flow_volume_score', 'cramps_score', 'sorebreasts_score', 
    'flow_color'
]

feature_sets = {
    "Baseline": baseline_cols,
    "Forward": forward_features,
    "Backward": backward_features,
    "ANOVA": anova_features,
    "SelectKBest": kbest_features
}

# 3. Define Models and Param Grids
models_config = {
    "XGBoost": {
        "model": xgb.XGBClassifier(objective='multi:softprob', eval_metric='mlogloss', random_state=42),
        "params": {
            'classifier__n_estimators': [100, 200],
            'classifier__learning_rate': [0.05, 0.1],
            'classifier__max_depth': [3, 5]
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42, class_weight='balanced'),
        "params": {
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [None, 10],
            'classifier__min_samples_split': [2, 5]
        }
    },
    "LogisticRegression": {
        # Increased max_iter to prevent convergence warnings
        "model": LogisticRegression(random_state=42, class_weight='balanced', max_iter=3000, solver='lbfgs'),
        "params": {
            'classifier__C': [0.1, 1, 10],
        }
    },
    "DecisionTree": {
        "model": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
        "params": {
            'classifier__max_depth': [5, 10, None],
            'classifier__min_samples_split': [2, 10]
        }
    }
}

# 4. Master Loop
results_list = []

print(f"{'Feature Set':<15} | {'Model':<20} | {'Status':<10}")
print("-" * 50)

for fs_name, features in feature_sets.items():
    # Prepare Data
    X_subset = df[features].copy()
    y = df[TARGET_COL]
    
    # Encode Target
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )
    
    # Preprocessor (Fixed: Now includes Imputation!)
    num_cols = X_subset.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = X_subset.select_dtypes(include=['object', 'category']).columns
    
    # Pipeline for Numeric: Impute -> Scale
    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ('scaler', StandardScaler())
    ])
    
    # Pipeline for Categorical: Impute -> Encode
    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    preprocessor = ColumnTransformer(transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])
    
    for model_name, config in models_config.items():
        print(f"{fs_name:<15} | {model_name:<20} | Tuning...", end="\r")
        
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', config['model'])
        ])
        
        grid = GridSearchCV(
            pipeline, 
            config['params'], 
            cv=3, 
            scoring='accuracy', 
            n_jobs=-1
        )
        
        try:
            grid.fit(X_train, y_train)
            
            # Test Score
            y_pred = grid.predict(X_test)
            test_acc = accuracy_score(y_test, y_pred)
            
            results_list.append({
                "Feature Set": fs_name,
                "Model": model_name,
                "Best CV Accuracy": grid.best_score_,
                "Test Accuracy": test_acc,
                "Best Params": str(grid.best_params_)
            })
            print(f"{fs_name:<15} | {model_name:<20} | Done (Acc: {test_acc:.4f})")
            
        except Exception as e:
            print(f"{fs_name:<15} | {model_name:<20} | FAILED: {str(e)}")

# 5. Display Leaderboard
results_df = pd.DataFrame(results_list)
if not results_df.empty:
    results_df = results_df.sort_values(by="Test Accuracy", ascending=False)
    print("\n=== FINAL TUNED LEADERBOARD ===")
    print(results_df[['Feature Set', 'Model', 'Test Accuracy', 'Best CV Accuracy']].to_string(index=False))
else:
    print("No results collected due to errors.")

Feature Set     | Model                | Status    
--------------------------------------------------
Baseline        | XGBoost              | Done (Acc: 0.6987)
Baseline        | RandomForest         | Done (Acc: 0.6801)
Baseline        | LogisticRegression   | Done (Acc: 0.6010)
Baseline        | DecisionTree         | Done (Acc: 0.5421)
Forward         | XGBoost              | Done (Acc: 0.6347)
Forward         | RandomForest         | Done (Acc: 0.6582)
Forward         | LogisticRegression   | Done (Acc: 0.5707)
Forward         | DecisionTree         | Done (Acc: 0.4916)
Backward        | XGBoost              | Done (Acc: 0.6397)
Backward        | RandomForest         | Done (Acc: 0.6549)
Backward        | LogisticRegression   | Done (Acc: 0.5589)
Backward        | DecisionTree         | Done (Acc: 0.5404)
ANOVA           | XGBoost              | Done (Acc: 0.6246)
ANOVA           | RandomForest         | Done (Acc: 0.6431)
ANOVA           | LogisticRegression   | Done (Acc: 0.570